In [38]:
import os
import pickle
import pandas as pd
import cloudpickle
import mlflow
from dotenv import load_dotenv
from mlflow.tracking import MlflowClient
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer

In [39]:
# Load environment variables from .env file
dotenv_path = r"D:\RGT\Code\MLOps\Orchestration\aws.env"
load_dotenv(dotenv_path)

# Access AWS credentials from environment variables
aws_access_key_id = os.getenv("access_key_id")
aws_secret_access_key = os.getenv("secret_access_key")
aws_region = os.getenv("region")

# Set AWS credentials as environment variables
os.environ['AWS_ACCESS_KEY_ID'] = aws_access_key_id
os.environ['AWS_SECRET_ACCESS_KEY'] = aws_secret_access_key
os.environ['AWS_REGION'] = aws_region

In [40]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("green-taxi-duration")

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1714640074745, experiment_id='3', last_update_time=1714640074745, lifecycle_stage='active', name='green-taxi-duration', tags={}>

In [41]:
def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df


def prepare_dictionaries(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dicts

In [42]:
df_train = read_dataframe(r'D:\RGT\Code\MLOps\data\green_tripdata_2021-01.parquet')
df_val = read_dataframe(r'D:\RGT\Code\MLOps\data\green_tripdata_2021-02.parquet')

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

dict_train = prepare_dictionaries(df_train)
dict_val = prepare_dictionaries(df_val)

In [43]:
with mlflow.start_run(run_id="c3379ddad0034e82b039283c3ebfbba9"):
    params = dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=0)
    mlflow.log_params(params)

    pipeline = make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs=-1)
    )

    pipeline.fit(dict_train, y_train)
    y_pred = pipeline.predict(dict_val)

    rmse = mean_squared_error(y_pred, y_val, squared=False)
    print(params, rmse)
    mlflow.log_metric('rmse', rmse)
    
    # Serialize and log DictVectorizer
    with open("model/dict_vectorizer.bin", "wb") as f:
        cloudpickle.dump(pipeline.named_steps['dictvectorizer'], f)
    mlflow.log_artifact("model/dict_vectorizer.bin")

    # Log model with artifact path
    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="model",
        registered_model_name=None,
        conda_env=None,
        serialization_format='cloudpickle',
        signature=None,
        input_example=None
    )

{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 0} 6.7558229919200725


d:\RGT\Code\rgtenv\lib\site-packages\_distutils_hack\__init__.py:11: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
d:\RGT\Code\rgtenv\lib\site-packages\_distutils_hack\__init__.py:26: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


In [44]:
MLFLOW_TRACKING_URI = 'http://127.0.0.1:5000'
RUN_ID = 'c3379ddad0034e82b039283c3ebfbba9'

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [45]:
# List artifacts for the specified run
artifacts = client.list_artifacts(run_id=RUN_ID)

In [46]:
client.download_artifacts(run_id=RUN_ID, path="model")

'C:\\Users\\SAMBER~1\\AppData\\Local\\Temp\\tmprdqo_9po\\model'

In [47]:
for artifact in artifacts:
    print(artifact.path)

dict_vectorizer.bin
model


In [48]:
directory = "D:\RGT\Code\MLOps\Deployment\web-service-mlflow"
path = client.download_artifacts(run_id=RUN_ID, path='model', dst_path=directory)

In [49]:
path = "model/dict_vectorizer.bin" 
artifact_path = os.path.join(directory, path)

In [50]:
# Now you can load the downloaded pandas object using read_pickle
dv = pd.read_pickle(artifact_path) # To be debugged later

In [51]:
dv

DictVectorizer()